# 02 – ResNet50 Feature Extraction (HAM10000)

Extracts 2048-dimensional feature vectors from a pretrained ResNet50
(ImageNet weights, final FC layer removed) for every image in the dataset,
preserving the existing leakage-safe `train / val / test` split from
`results/splits.csv`.

**Outputs**
- `results/label_mapping.json` – `{dx_string: int_label}` for 7 classes
- `results/resnet50_features.npz` – X_train, y_train, X_val, y_val, X_test, y_test

## 0  Imports & Setup

In [ ]:
import json
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from PIL import Image

warnings.filterwarnings('ignore')

ROOT        = Path('..').resolve()
RESULTS_DIR = ROOT / 'results'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

SPLITS_CSV       = RESULTS_DIR / 'splits.csv'
LABEL_MAP_JSON   = RESULTS_DIR / 'label_mapping.json'
FEATURES_NPZ     = RESULTS_DIR / 'resnet50_features.npz'

BATCH_SIZE       = 16
IMG_SIZE         = 224
LOG_EVERY        = 20   # print progress every N batches

# Device — CPU only (no CUDA on this machine)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
print(f'PyTorch version: {torch.__version__}')

## 1  Load splits.csv & Encode Labels

In [ ]:
splits = pd.read_csv(SPLITS_CSV)
print(f'splits.csv loaded: {len(splits):,} rows')
print(f'Columns: {list(splits.columns)}')
print()
print('Split counts (images):')
print(splits['split'].value_counts()[['train', 'val', 'test']].to_string())
splits.head(3)

In [ ]:
# Fixed 7-class label mapping (alphabetical for reproducibility)
CLASSES = sorted(['akiec', 'bcc', 'bkl', 'df', 'mel', 'nv', 'vasc'])
label_mapping = {cls: idx for idx, cls in enumerate(CLASSES)}

print('Label mapping (dx -> int):')
for k, v in label_mapping.items():
    print(f'  {k:6s} -> {v}')

with open(LABEL_MAP_JSON, 'w') as f:
    json.dump(label_mapping, f, indent=2)
print(f'\nSaved: {LABEL_MAP_JSON}')

splits['label'] = splits['dx'].map(label_mapping)
assert splits['label'].isna().sum() == 0, 'Unmapped dx values found!'
print('All dx values mapped successfully.')

## 2  Sanity-Check: Per-Class Counts Per Split

In [ ]:
pivot = (
    splits.groupby(['dx', 'split'])
    .size()
    .unstack(fill_value=0)
    [['train', 'val', 'test']]
)
pivot['total'] = pivot.sum(axis=1)
print('Per-class image counts per split (from splits.csv):')
print(pivot.to_string())
print()
totals = splits['split'].value_counts()[['train', 'val', 'test']]
print(f'Total -> train={totals["train"]:,}  val={totals["val"]:,}  test={totals["test"]:,}')
assert totals['train'] == 7054, f'Expected 7054 train images, got {totals["train"]}'
assert totals['val']   == 1464, f'Expected 1464 val images, got {totals["val"]}'
assert totals['test']  == 1497, f'Expected 1497 test images, got {totals["test"]}'
print('[OK] Split counts match expected: train=7054, val=1464, test=1497')

## 3  Build Dataset & DataLoader

In [ ]:
# Standard ImageNet normalisation
imagenet_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std =[0.229, 0.224, 0.225]
    ),
])


class SkinLesionDataset(Dataset):
    """Simple dataset that reads images from filepath column."""

    def __init__(self, df: pd.DataFrame, transform=None):
        self.paths     = df['filepath'].tolist()
        self.labels    = df['label'].tolist()
        self.image_ids = df['image_id'].tolist()
        self.transform = transform

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        img = Image.open(self.paths[idx]).convert('RGB')
        if self.transform:
            img = self.transform(img)
        return img, self.labels[idx]


def make_loader(df_split: pd.DataFrame) -> DataLoader:
    ds = SkinLesionDataset(df_split, transform=imagenet_transform)
    return DataLoader(ds, batch_size=BATCH_SIZE, shuffle=False,
                      num_workers=0, pin_memory=False)


df_train = splits[splits['split'] == 'train'].reset_index(drop=True)
df_val   = splits[splits['split'] == 'val'].reset_index(drop=True)
df_test  = splits[splits['split'] == 'test'].reset_index(drop=True)

train_loader = make_loader(df_train)
val_loader   = make_loader(df_val)
test_loader  = make_loader(df_test)

print(f'DataLoaders ready')
print(f'  train : {len(df_train):,} images -> {len(train_loader):,} batches')
print(f'  val   : {len(df_val):,} images   -> {len(val_loader):,} batches')
print(f'  test  : {len(df_test):,} images   -> {len(test_loader):,} batches')

## 4  Load Pretrained ResNet50 (Head Removed)

In [ ]:
# Load ResNet50 with ImageNet weights
backbone = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V1)

# Remove final FC layer -> output is 2048-d avg-pool vector
feature_extractor = nn.Sequential(*list(backbone.children())[:-1])
feature_extractor = feature_extractor.to(device)
feature_extractor.eval()   # freeze BN layers in eval mode

# Count parameters (frozen — not training anything)
total_params = sum(p.numel() for p in feature_extractor.parameters())
print(f'ResNet50 backbone loaded (ImageNet weights)')
print(f'  Parameters  : {total_params:,}')
print(f'  Output dim  : 2048  (global average-pooled)')
print(f'  Mode        : eval / frozen (feature extraction only)')
print(f'  Device      : {device}')

## 5  Feature Extraction Function

In [ ]:
def extract_features(loader: DataLoader, split_name: str):
    """Run inference through the backbone and collect feature vectors + labels."""
    all_features = []
    all_labels   = []
    total_batches = len(loader)
    t0 = time.time()

    print(f'[{split_name}] Extracting features from {len(loader.dataset):,} images'
          f' ({total_batches} batches x batch_size={BATCH_SIZE})...')

    with torch.no_grad():
        for batch_idx, (imgs, labels) in enumerate(loader):
            imgs = imgs.to(device)

            # Forward pass -> shape: (B, 2048, 1, 1)
            feats = feature_extractor(imgs)
            # Flatten -> shape: (B, 2048)
            feats = feats.view(feats.size(0), -1)

            all_features.append(feats.cpu().numpy())
            all_labels.append(labels.numpy() if isinstance(labels, torch.Tensor)
                              else np.array(labels))

            if (batch_idx + 1) % LOG_EVERY == 0 or (batch_idx + 1) == total_batches:
                elapsed = time.time() - t0
                imgs_done = min((batch_idx + 1) * BATCH_SIZE, len(loader.dataset))
                print(f'  Batch {batch_idx+1:>4d}/{total_batches}'
                      f'  |  {imgs_done:>5,}/{len(loader.dataset):,} images'
                      f'  |  {elapsed:>6.1f}s elapsed')

    X = np.vstack(all_features)       # (N, 2048)
    y = np.concatenate(all_labels)    # (N,)

    elapsed_total = time.time() - t0
    print(f'[{split_name}] Done in {elapsed_total:.1f}s  ->  X.shape={X.shape},  y.shape={y.shape}')
    print()
    return X, y

print('extract_features() defined.')

## 6  Run Extraction — Train Split

In [ ]:
X_train, y_train = extract_features(train_loader, 'TRAIN')
print(f'Shape confirmation  X_train: {X_train.shape}  y_train: {y_train.shape}')

## 7  Run Extraction — Val Split

In [ ]:
X_val, y_val = extract_features(val_loader, 'VAL')
print(f'Shape confirmation  X_val:   {X_val.shape}    y_val:   {y_val.shape}')

## 8  Run Extraction — Test Split

In [ ]:
X_test, y_test = extract_features(test_loader, 'TEST')
print(f'Shape confirmation  X_test:  {X_test.shape}    y_test:  {y_test.shape}')

## 9  Save All Arrays to .npz

In [ ]:
np.savez_compressed(
    FEATURES_NPZ,
    X_train=X_train, y_train=y_train,
    X_val=X_val,     y_val=y_val,
    X_test=X_test,   y_test=y_test,
)

size_mb = FEATURES_NPZ.stat().st_size / (1024 ** 2)
print(f'Saved: {FEATURES_NPZ}')
print(f'File size: {size_mb:.1f} MB')
print()

# Reload and verify shapes
loaded = np.load(FEATURES_NPZ)
print('Verification (reloaded from disk):')
for key in ['X_train', 'y_train', 'X_val', 'y_val', 'X_test', 'y_test']:
    arr = loaded[key]
    print(f'  {key:10s}  shape={str(arr.shape):20s}  dtype={arr.dtype}')
loaded.close()

## 10  Final Summary

In [ ]:
print('=' * 60)
print('ResNet50 Feature Extraction - Summary')
print('=' * 60)
print(f'Model        : ResNet50 (ImageNet weights, head removed)')
print(f'Feature dim  : 2048')
print(f'Image size   : {IMG_SIZE}x{IMG_SIZE} px (resized)')
print(f'Normalisation: ImageNet mean/std')
print(f'Device       : {device}')
print()
print(f'Arrays saved to: {FEATURES_NPZ}')
print(f'  X_train : {X_train.shape}  y_train : {y_train.shape}')
print(f'  X_val   : {X_val.shape}    y_val   : {y_val.shape}')
print(f'  X_test  : {X_test.shape}   y_test  : {y_test.shape}')
print()
print(f'Label mapping: {LABEL_MAP_JSON}')
for k, v in label_mapping.items():
    print(f'  {v}  {k}')
print()
print('[OK] Feature extraction complete. Ready for downstream classifiers.')